# Assortative mixing

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/14-assortative-mixing.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

People who share a trait often contact each other more than they contact
other groups (**assortative** mixing). This chapter contrasts three density-
dependent SIRs: unstratified, stratified with a homogeneous matrix, and
stratified with an assortative matrix — then shows why assortativity only
splits group-specific epidemics once groups also differ epidemiologically
(here, infectiousness via
{meth}`~summer4.epi.EpiModel.add_infectiousness_adjustments`).



## Starting assumptions

We work under **density-dependent** transmission so matrix entries read as
per-capita×per-capita contact weights, and we keep susceptibility and
infectiousness uniform until the last section. With two groups the matrix is
$2\times 2$; symmetry of opposite off-diagonal cells is natural when the
entries are undirected contact rates.

An assortative matrix puts larger weight on the diagonal than the
off-diagonal — more within-group than between-group contact.



In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from summer4 import (
    Compartments,
    GroupedOutput,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
)
from summer4.epi import EpiModel

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"


def assortative_matrix(intergroup: float) -> np.ndarray:
    """Ones on the diagonal; ``intergroup`` on the off-diagonal."""
    return np.array(
        [
            [1.0, intergroup],
            [intergroup, 1.0],
        ],
        dtype=float,
    )


K_show = assortative_matrix(0.5)
assert K_show[0, 0] == 1.0 and K_show[0, 1] == 0.5
px.imshow(
    K_show,
    x=["group1", "group2"],
    y=["group1", "group2"],
    title="Assortative mixing (intergroup = 0.5)",
    labels={"color": "weight"},
)


## Unstratified → homogeneous → assortative

Build the same density-dependent SIR three ways. Unstratified models still
need a singleton mixing axis for `ForceOfInfection` (the dummy `pop`
pattern from {doc}`09-transmission-assumptions`). Stratified models use a
`group` property; population shares go into `y0` rather than a separate
split API.

Reducing off-diagonal contacts also reduces mean daily risk. With equal
group sizes and intergroup weight $0.5$, each row averages to $0.75$, so we
scale the contact rate by $4/3$ to keep the same overall force as the
ones-matrix case.



In [ ]:
state = Property("state", ("susceptible", "infectious", "recovered"))
pop = Property("pop", ("all",))
group = Property("group", ("group1", "group2"))
pmap_u = PropertyMap.from_property(state).stratify(pop)
pmap_g = PropertyMap.from_property(state).stratify(group)

END_TIME = 40.0
CONTACT = 0.5
RECOVERY = 1.0 / 4.0
POPULATION = 1.0
SEED = 0.01
PROP1 = 0.5

PLAN_U = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, END_TIME, int(END_TIME * 5) + 1),
)
PLAN_G = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "foi": SaveRequest(GroupedOutput("infection")),
    },
    ts=np.linspace(0.0, END_TIME, int(END_TIME * 5) + 1),
)


def y0_unstratified() -> np.ndarray:
    y0 = np.zeros(pmap_u.size)
    y0[pmap_u.select(state["susceptible"])] = POPULATION - SEED
    y0[pmap_u.select(state["infectious"])] = SEED
    return y0


def y0_groups(*, seed1: float, seed2: float) -> np.ndarray:
    y0 = np.zeros(pmap_g.size)
    sus = POPULATION - seed1 - seed2
    y0[pmap_g.select(state["susceptible"] & group["group1"])] = PROP1 * sus
    y0[pmap_g.select(state["susceptible"] & group["group2"])] = (1.0 - PROP1) * sus
    y0[pmap_g.select(state["infectious"] & group["group1"])] = seed1
    y0[pmap_g.select(state["infectious"] & group["group2"])] = seed2
    return y0


def build_and_run(
    pmap: PropertyMap,
    mix_prop: Property,
    matrix: np.ndarray,
    y0: np.ndarray,
    plan: SavePlan,
    *,
    contact: float = CONTACT,
    infectiousness: dict[str, float] | None = None,
):
    model = EpiModel(pmap, infectious=state["infectious"])
    model.set_mixing_matrix(
        mix_prop, matrix, normalize="none", check_reciprocal=False
    )
    if infectiousness is not None:
        model.add_infectiousness_adjustments(
            mix_prop, infectiousness, normalize=None
        )
    model.add_infection_density_flow(
        "infection",
        state["susceptible"],
        state["infectious"],
        contact,
    )
    model.add_transition_flow(
        "recovery", state["infectious"], state["recovered"], RECOVERY
    )
    return model.compile().run(
        {}, y0, t0=0.0, t1=END_TIME, dt=0.1, save=plan, solver="euler"
    )


def overall_prevalence(res) -> pd.Series:
    i = np.asarray(res["comp"].select(state["infectious"]).values.data)
    return pd.Series(
        i.sum(axis=1) / POPULATION,
        index=np.asarray(res["comp"].times.values),
    )


def group_prevalence(res) -> pd.DataFrame:
    frame = res["comp"].select(state["infectious"]).to_pandas()
    # Labels look like ``state=infectious_group=group1``.
    frame.columns = [c.split("=")[-1] for c in frame.columns]
    sizes = {"group1": PROP1 * POPULATION, "group2": (1.0 - PROP1) * POPULATION}
    for col in frame.columns:
        frame[col] = frame[col] / sizes[col]
    return frame


res_u = build_and_run(pmap_u, pop, np.array([[1.0]]), y0_unstratified(), PLAN_U)
y0_eq = y0_groups(seed1=SEED / 2, seed2=SEED / 2)
K_hom = np.ones((2, 2))
K_ass = assortative_matrix(0.5)
res_hom = build_and_run(pmap_g, group, K_hom, y0_eq, PLAN_G)
res_ass = build_and_run(
    pmap_g, group, K_ass, y0_eq, PLAN_G, contact=CONTACT * (4.0 / 3.0)
)

compare = pd.DataFrame(
    {
        "unstratified": overall_prevalence(res_u),
        "homogeneous stratified": overall_prevalence(res_hom),
        "assortative stratified": overall_prevalence(res_ass),
    }
)
np.testing.assert_allclose(
    compare["unstratified"].values,
    compare["homogeneous stratified"].values,
    atol=1e-5,
)
# Symmetric groups + assortative mixing alone ⇒ overall curve matches homogeneous
# after contact rescaling; mixing structure does not yet split dynamics.
np.testing.assert_allclose(
    compare["homogeneous stratified"].values,
    compare["assortative stratified"].values,
    atol=1e-4,
)

compare.plot(
    title="Overall prevalence: mixing alone does not change the epidemic",
    labels={"index": "time", "value": "prevalence"},
)


## Assortativity matters once groups differ

Double infectiousness in `group1`. Under a **homogeneous** matrix every
group still sees the same $\lambda(t)$, so group-specific prevalence stays
aligned. Under an **assortative** matrix, the more infectious group also
mixes preferentially with itself, and prevalence separates.



In [ ]:
inf_weights = {"group1": 2.0, "group2": 1.0}

res_hom_adj = build_and_run(
    pmap_g, group, K_hom, y0_eq, PLAN_G, infectiousness=inf_weights
)
res_ass_adj = build_and_run(
    pmap_g,
    group,
    K_ass,
    y0_eq,
    PLAN_G,
    contact=CONTACT * (4.0 / 3.0),
    infectiousness=inf_weights,
)

foi_hom = res_hom_adj["foi"].to_pandas()
foi_hom.columns = list(group.traits)
foi_ass = res_ass_adj["foi"].to_pandas()
foi_ass.columns = list(group.traits)

d_hom = float(np.max(np.abs(foi_hom["group1"] - foi_hom["group2"])))
d_ass = float(np.max(np.abs(foi_ass["group1"] - foi_ass["group2"])))
assert d_hom == 0.0, "homogeneous mixing collapses λ across groups"
assert d_ass > 0.0, "assortative mixing must produce group-specific λ"
print(f"homogeneous max |Δλ| = {d_hom}")
print(f"assortative max |Δλ| = {d_ass:.4f}")

prev_ass = group_prevalence(res_ass_adj)
assert float(prev_ass["group1"].max()) > float(prev_ass["group2"].max())

prev_ass.plot(
    title="Group prevalence under assortative mixing + higher group1 infectiousness",
    labels={"index": "time", "value": "prevalence within group"},
)


### Extremes

With intergroup contact set to zero the groups decouple: `group1` effectively
runs an SIR with twice the infectiousness of `group2`.



In [ ]:
res_decoupled = build_and_run(
    pmap_g,
    group,
    assortative_matrix(0.0),
    y0_eq,
    PLAN_G,
    infectiousness=inf_weights,
)
prev_dec = group_prevalence(res_decoupled)
assert float(prev_dec["group1"].max()) > float(prev_dec["group2"].max())
foi_dec = res_decoupled["foi"].to_pandas()
foi_dec.columns = list(group.traits)
assert float(foi_dec.iloc[5]["group1"]) > float(foi_dec.iloc[5]["group2"])

prev_dec.plot(
    title="Decoupled groups (intergroup = 0) with group1 twice as infectious",
    labels={"index": "time", "value": "prevalence within group"},
)
